# EXP-013 (red-team): forged evidence, reordered leaves, forged parent links, missing receipts — deterministic gates

## 1 Requirements and predeclared hypothesis
H0: every listed tamper is rejected by the verifiers in this repo, and the one known collision (duplicate-last) is bound by leaf_count. No model involved.

## 2 Input source, license, hashes
evidence/freeze/** metadata, evidence/sources/supabase_source_intake.json, evidence/stats/discovery_pair_flags.jsonl.

## 3 Frozen settings
same Merkle scheme; verifiers scripts/verify_predecessor.py, scripts/validate_pair_export.py, scripts/redteam/redteam_verify_sources.py.

## 4 Deterministic preprocessing
in-memory copies only; nothing on disk is modified.

## 5 Model/harness execution
none.

## 6 Analysis
attack matrix: expected REJECT vs observed.

## 7 Negative controls
the untampered inputs must PASS in the same cells.

## 8 Outputs
exp013_attacks.json.

## 9 FCO/FCG dependency changes
none.

## 10 Claim ceiling
Notebook PASS verifies the local replay only. It never upgrades NOT_TESTED, BLOCKED or NEGATIVE states, and it never attributes a causal gain to a hash, leaf or edge: hashes bind bytes, controlled interventions bind effects.


In [1]:
import json, hashlib, sys
from pathlib import Path
R = Path.cwd(); sys.path.insert(0, str(R / 'src')); sys.path.insert(0, str(R / 'scripts/redteam'))
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def jsonl(p): return [json.loads(x) for x in Path(p).read_text().splitlines() if x.strip()]
OUT = R / 'evidence/notebook_runs_redteam_v1/outputs'; OUT.mkdir(parents=True, exist_ok=True)
def emit(name, obj):
    p = OUT / name; p.write_text(json.dumps(obj, sort_keys=True, indent=2) + '\n'); print('OUTPUT', name, sha(p)); return p
import tempfile, shutil, copy
from redteam_verify_sources import leaf, root, proof_ok
from verify_predecessor import verify
from validate_pair_export import check
attacks = {}
pred = R / 'evidence/freeze/predecessor'
rows = jsonl(pred / 'leaves.jsonl'); man = json.loads((pred / 'manifest.json').read_text())
def run_verify(mutate_rows=None, mutate_man=None):
    with tempfile.TemporaryDirectory() as td:
        d = Path(td); rr = copy.deepcopy(rows); mm = copy.deepcopy(man)
        if mutate_rows: mutate_rows(rr)
        if mutate_man: mutate_man(mm)
        (d / 'leaves.jsonl').write_text(''.join(json.dumps(x, sort_keys=True, separators=(',', ':')) + '\n' for x in rr)); (d / 'manifest.json').write_text(json.dumps(mm))
        try: verify(d); return 'PASS'
        except AssertionError as e: return 'REJECT:' + str(e)[:40]
attacks['untampered_control'] = run_verify()
attacks['forged_source_leaf_sha'] = run_verify(lambda rr: rr[4].__setitem__('sha256', 'f' * 64))
attacks['reordered_leaves'] = run_verify(lambda rr: rr.__setitem__(slice(2, 4), [rr[3], rr[2]]))
attacks['removed_leaf'] = run_verify(lambda rr: rr.pop(5))
attacks['forged_parent_root_in_context'] = run_verify(lambda rr: rr[0]['context']['parent'].__setitem__('root', '0' * 64))
attacks['forged_edge_status'] = run_verify(lambda rr: rr[-1]['edges'][2].__setitem__('status', 'EXECUTED'))
attacks['duplicate_last_leaf_appended'] = run_verify(lambda rr: rr.append(copy.deepcopy(rr[-1])))
attacks['duplicate_last_leaf_and_count_forged'] = run_verify(lambda rr: rr.append(copy.deepcopy(rr[-1])), lambda mm: (mm.__setitem__('leaf_count', 16), mm['leaf_order'].append(mm['leaf_order'][-1])))
assert attacks['untampered_control'] == 'PASS'
assert all(v.startswith('REJECT') for k, v in attacks.items() if k not in ('untampered_control', 'duplicate_last_leaf_and_count_forged'))
# F-01 ceiling: duplicating the last leaf AND forging leaf_count/leaf_order keeps root and passes the unsigned verifier.
attacks['F01_note'] = 'duplicate_last_leaf_and_count_forged=' + attacks['duplicate_last_leaf_and_count_forged'] + ' -> count binding must live inside a hashed leaf or a signature (fixed in fco.redteam_successor.v1 context leaf)'
# pair export attacks
def pair_attack(mut):
    with tempfile.TemporaryDirectory() as td:
        t = Path(td); (t / 'evidence/stats').mkdir(parents=True); (t / 'evidence/freeze').mkdir(parents=True)
        shutil.copy(R / 'evidence/freeze/longebench_split_manifest.json', t / 'evidence/freeze/longebench_split_manifest.json')
        pr = jsonl(R / 'evidence/stats/discovery_pair_flags.jsonl'); mut(pr)
        (t / 'evidence/stats/discovery_pair_flags.jsonl').write_text(''.join(json.dumps(x) + '\n' for x in pr))
        try: check(t); return 'PASS'
        except AssertionError as e: return 'REJECT:' + str(e)[:40]
split = json.loads((R / 'evidence/freeze/longebench_split_manifest.json').read_text())
heldout_key = next(iter(next(iter(split['task_partitions'].values()))['heldout']))
attacks['pairs_untampered_control'] = pair_attack(lambda pr: None)
attacks['pairs_heldout_leak'] = pair_attack(lambda pr: pr[0].__setitem__('case_digest_sha256', heldout_key))
attacks['pairs_duplicate_case'] = pair_attack(lambda pr: pr.__setitem__(1, dict(pr[0])))
attacks['pairs_wrong_task_binding'] = pair_attack(lambda pr: pr[0].__setitem__('task', 'LB-0315' if pr[0]['task'] != 'LB-0315' else 'LB-0307'))
attacks['pairs_dropped_case'] = pair_attack(lambda pr: pr.pop())
assert attacks['pairs_untampered_control'] == 'PASS' and all(v.startswith('REJECT') for k, v in attacks.items() if k.startswith('pairs_') and 'control' not in k)
# supabase gate must still fail closed
sup = json.loads((R / 'evidence/sources/supabase_source_intake.json').read_text())
attacks['supabase_live_rows_as_benchmark'] = 'REJECT:state=' + sup['state'] + ';root=' + sup['snapshot_merkle_root']
emit('exp013_attacks.json', attacks); print(json.dumps(attacks, indent=2))

ModuleNotFoundError: No module named 'verify_predecessor'